In [19]:
# Add markdown and try to write neat and clean code.
import numpy as np 
import pandas as pd
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import sys
from sklearn.model_selection import train_test_split


In [20]:
import kagglehub
suchintikasarkar_sentiment_analysis_for_mental_health_path = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')

print("Printing the complete kaggle dataSet.")

Printing the complete kaggle dataSet.


In [21]:
nlp = None
stop_words = set()
lemmatizer = None

try:
    nltk.download('wordnet', quiet = True, raise_on_error = True)
    nltk.download('stopwords', quiet = True, raise_on_error = True)
    nltk.download('punkt', quiet=True, raise_on_error=True)

    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    _ = lemmatizer.lemmatize("test") # Dummy call to ensure WordNet is properly loaded
    print("NLTK resources (stopwords, wordnet) loaded successfully.")

except Exception as e:
    print(f"NLTK resource download or initialization error: {e}.")
    print("Stopword removal or NLTK Lemmatization might be affected or skipped.")

try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model 'en_core_web_sm' loaded successfully.")
except Exception as e:
    print(f"spaCy model ('en_core_web_sm') loading error: {e}. spaCy-based lemmatization will not be available.")
    nlp = None # Ensure nlp is None if loading fails

NLTK resources (stopwords, wordnet) loaded successfully.
spaCy model 'en_core_web_sm' loaded successfully.


In [22]:
print("\n Data Loading...")
df = pd.DataFrame()

DATA_PATH = '/kaggle/input/sentiment-analysis-for-mental-health/Combined Data.csv' # Replace with your actual path
TEXT_COLUMN = 'statement'
LABEL_COLUMN = 'status'

TEST_SIZE = 0.2  ## Test size is 20%

# General
SEED = 42

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"Error: Dataset not found at '{DATA_PATH}'. Using dummy data for demonstration.")
    # Dummy data for robust execution if actual data is missing
    df = pd.DataFrame({
        TEXT_COLUMN: [
            "I feel incredibly sad and overwhelmed today #depression",
            "This is a fantastic day, I'm so happy and energetic! www.example.com",
            "Just a regular, normal kind of day. Nothing special to report @user.",
            "Feeling very down and can't seem to cope with anything!!! #anxiety",
            "Excited about the future and all the possibilities! Life is good.",
            "LOL, this is gr8! I can't stop laughing.",
            "My anxiety peaks whenever I think about exams. It's debilitating.",
            "Making progress with my therapy for bipolar disorder. It's a journey.",
            "Sometimes the stress is just too much to handle. I need a break.",
            "Suicidal ideation is a heavy burden. Seeking help is important.",
            "Living with a personality disorder presents unique daily challenges."
        ],
        LABEL_COLUMN: ["Depression", "Normal", "Normal", "Anxiety", "Normal", "Normal",
                       "Anxiety", "Bipolar", "Stress", "Suicidal", "Personality disorder"]
    })

# Clean column names (e.g., remove leading/trailing spaces)
df.columns = df.columns.str.strip()

if 'Unnamed: 0' in df.columns: # Remove common index column from CSVs
    df = df.drop('Unnamed: 0', axis=1)

# Ensure text and label columns exist
if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
    sys.exit(f"Critical Error: Text column ('{TEXT_COLUMN}') or Label column ('{LABEL_COLUMN}') not found in the data.")

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).fillna('') # Convert text to string, fill NaNs
df = df.dropna(subset=[LABEL_COLUMN]) # Remove rows where the label itself is NaN
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str) # Convert labels to string

unique_labels = sorted(df[LABEL_COLUMN].unique().tolist())
if not unique_labels:
    sys.exit(f"Critical Error: No unique labels found in column '{LABEL_COLUMN}' after processing. Please check your data.")

label_map = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_map.items()}
df['label_id'] = df[LABEL_COLUMN].map(label_map) # Create integer labels
print(f"Found {len(unique_labels)} unique labels: {unique_labels}")
print(f"Label map: {label_map}")

X = df[TEXT_COLUMN] # Features
y = df['label_id']  # Target

if len(y.unique()) < 2: # Need at least two classes for classification
    sys.exit("Critical Error: Not enough unique classes in the label column for training (need at least 2).")

# Initial split into training and test sets
try:
    X_train_full, X_test_raw, y_train_full, y_test_raw = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
except ValueError: # Fallback if stratification fails (e.g., a class has < 2 members for splitting)
    print("Warning: Stratification failed for initial train/test split. Proceeding without stratification.")
    X_train_full, X_test_raw, y_train_full, y_test_raw = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED
    )

print(f"Initial full training set size: {len(X_train_full)}, Raw test set size: {len(X_test_raw)}")
print("Data Loading and Initial Splitting Complete.")


 Data Loading...
Error: Dataset not found at '/kaggle/input/sentiment-analysis-for-mental-health/Combined Data.csv'. Using dummy data for demonstration.
Found 7 unique labels: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']
Label map: {'Anxiety': 0, 'Bipolar': 1, 'Depression': 2, 'Normal': 3, 'Personality disorder': 4, 'Stress': 5, 'Suicidal': 6}
Initial full training set size: 8, Raw test set size: 3
Data Loading and Initial Splitting Complete.
